<a href="https://colab.research.google.com/github/justinliu00/MNIST-CNN-TRAIN/blob/main/tests/MNIST_t2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

In [59]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize images
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Flatten images from 28x28 -> 784
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# One-hot encode labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [60]:
# Input dimensions
input_width = 28
input_height = 28
input_channels = 1
input_pixels = 784

# Convolution layer parameters
n_conv1 = 32
n_conv2 = 64

stride_conv1 = 1
stride_conv2 = 1

conv1_k = 5
conv2_k = 5

max_pool1_k = 2
max_pool2_k = 2

# Fully connected layer
n_hidden = 1024
n_out = 10

# Calculate flattened size
input_size_to_hidden = (
    (input_width // (max_pool1_k * max_pool2_k)) *
    (input_height // (max_pool1_k * max_pool2_k)) *
    n_conv2
)

In [61]:
weights = {
    "wc1": tf.Variable(
        tf.random_normal([conv1_k, conv1_k, input_channels, n_conv1])
    ),

    "wc2": tf.Variable(
        tf.random_normal([conv2_k, conv2_k, n_conv1, n_conv2])
    ),

    "wh1": tf.Variable(
        tf.random_normal([input_size_to_hidden, n_hidden])
    ),

    "wo": tf.Variable(
        tf.random_normal([n_hidden, n_out])
    )
}

biases = {
    "bc1": tf.Variable(tf.random_normal([n_conv1])),
    "bc2": tf.Variable(tf.random_normal([n_conv2])),
    "bh1": tf.Variable(tf.random_normal([n_hidden])),
    "bo": tf.Variable(tf.random_normal([n_out]))
}

In [62]:
def conv(x, weights, bias, strides=1):

    out = tf.nn.conv2d(
        x,
        weights,
        padding="SAME",
        strides=[1, strides, strides, 1]
    )

    out = tf.nn.bias_add(out, bias)

    out = tf.nn.relu(out)

    return out


def maxpooling(x, k=2):

    return tf.nn.max_pool(
        x,
        padding="SAME",
        ksize=[1, k, k, 1],
        strides=[1, k, k, 1]
    )

In [63]:
def cnn(x, weights, biases, keep_prob):

    x = tf.reshape(
        x,
        shape=[-1, input_height, input_width, input_channels]
    )

    # Conv Layer 1
    conv1 = conv(
        x,
        weights['wc1'],
        biases['bc1'],
        stride_conv1
    )

    conv1_pool = maxpooling(conv1, max_pool1_k)

    # Conv Layer 2
    conv2 = conv(
        conv1_pool,
        weights['wc2'],
        biases['bc2'],
        stride_conv2
    )

    conv2_pool = maxpooling(conv2, max_pool2_k)

    # Flatten
    hidden_input = tf.reshape(
        conv2_pool,
        shape=[-1, input_size_to_hidden]
    )

    # Fully connected layer
    hidden_output_before_activation = tf.add(
        tf.matmul(hidden_input, weights['wh1']),
        biases['bh1']
    )

    hidden_output_before_dropout = tf.nn.relu(
        hidden_output_before_activation
    )

    # Dropout
    hidden_output = tf.nn.dropout(
        hidden_output_before_dropout,
        keep_prob
    )

    # Output layer
    output = tf.add(
        tf.matmul(hidden_output, weights['wo']),
        biases['bo']
    )

    return output

In [64]:
x = tf.placeholder("float", [None, input_pixels])

y = tf.placeholder(tf.float32, [None, n_out])

keep_prob = tf.placeholder("float")

pred = cnn(x, weights, biases, keep_prob)

In [65]:
cost = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits_v2(
        logits=pred,
        labels=y
    )
)

optimizer = tf.train.AdamOptimizer(learning_rate=0.01)

optimize = optimizer.minimize(cost)

In [66]:
sess = tf.Session()

sess.run(tf.global_variables_initializer())

In [67]:
batch_size = 100
num_examples = x_train.shape[0]

for i in range(25):

    total_cost = 0

    # Shuffle data
    indices = np.random.permutation(num_examples)

    x_train_shuffled = x_train[indices]
    y_train_shuffled = y_train[indices]

    num_batches = num_examples // batch_size

    for j in range(num_batches):

        start = j * batch_size
        end = start + batch_size

        batch_x = x_train_shuffled[start:end]
        batch_y = y_train_shuffled[start:end]

        c, _ = sess.run(
            [cost, optimize],
            feed_dict={
                x: batch_x,
                y: batch_y,
                keep_prob: 0.3
            }
        )

        total_cost += c

    print("Epoch:", i + 1, "Cost:", total_cost)

Epoch: 1 Cost: 1067431.4
Epoch: 2 Cost: 12101.52
Epoch: 3 Cost: 1462.6777
Epoch: 4 Cost: 1391.456
Epoch: 5 Cost: 1388.0625
Epoch: 6 Cost: 1361.2792
Epoch: 7 Cost: 1387.2202
Epoch: 8 Cost: 1374.0107
Epoch: 9 Cost: 1361.5624
Epoch: 10 Cost: 1389.8291
Epoch: 11 Cost: 1413.6562
Epoch: 12 Cost: 1404.8658
Epoch: 13 Cost: 1381.732
Epoch: 14 Cost: 1382.7042
Epoch: 15 Cost: 1381.1161
Epoch: 16 Cost: 1381.1293
Epoch: 17 Cost: 1381.1718
Epoch: 18 Cost: 1501.492
Epoch: 19 Cost: 1381.1853
Epoch: 20 Cost: 1381.2462
Epoch: 21 Cost: 1381.237
Epoch: 22 Cost: 1381.3514
Epoch: 23 Cost: 1423.0292
Epoch: 24 Cost: 1381.262
Epoch: 25 Cost: 1381.1824


In [71]:
predictions = tf.argmax(pred, 1)

correct_labels = tf.argmax(y, 1)

correct_predictions = tf.equal(
    predictions,
    correct_labels
)

predictions, correct_preds = sess.run(
    [predictions, correct_predictions],
    feed_dict={
        x: x_test,
        y: y_test,
        keep_prob: 1.0
    }
)

accuracy = correct_preds.sum() / y_test.shape[0]

print("Test accuracy:", accuracy)

Test accuracy: 0.1135
